In [1]:
!pip install pandas


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [46]:
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score,root_mean_squared_error

In [3]:
df = pd.read_csv("car-details.csv")
df.sample(5)

,name,company,model,edition,year,owner,fuel,seller_type,transmission,km_driven,mileage_mpg,engine_cc,max_power_bhp,torque_nm,seats,selling_price
3952,Maruti Swift AMT VXI,Maruti,Swift,AMT VXI,2019,First,Petrol,Individual,Automatic,7500,49.84,1197.0,81.8,113.0,5.0,600000
4443,Mahindra Bolero 2011-2019 SLX,Mahindra,Bolero,2011-2019 SLX,2013,First,Diesel,Individual,Manual,60000,37.50,2523.0,62.1,195.0,7.0,550000
582,Maruti Swift Dzire LDI,Maruti,Swift,Dzire LDI,2015,First,Diesel,Individual,Manual,170000,62.50,1248.0,74.0,190.0,5.0,370000
2964,Mahindra XUV500 W4,Mahindra,XUV500,W4,2014,First,Diesel,Individual,Manual,120000,35.50,2179.0,140.0,330.0,7.0,880000
2331,Maruti Ritz VXi (ABS) BS IV,Maruti,Ritz,VXi (ABS) BS IV,2011,First,Petrol,Individual,Manual,47000,43.47,1197.0,85.8,114.0,5.0,190000


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6926 entries, 0 to 6925
Data columns (total 16 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   name           6926 non-null   str    
 1   company        6926 non-null   str    
 2   model          6926 non-null   str    
 3   edition        6926 non-null   str    
 4   year           6926 non-null   int64  
 5   owner          6926 non-null   str    
 6   fuel           6926 non-null   str    
 7   seller_type    6926 non-null   str    
 8   transmission   6926 non-null   str    
 9   km_driven      6926 non-null   int64  
 10  mileage_mpg    6718 non-null   float64
 11  engine_cc      6718 non-null   float64
 12  max_power_bhp  6717 non-null   float64
 13  torque_nm      6717 non-null   float64
 14  seats          6718 non-null   float64
 15  selling_price  6926 non-null   int64  
dtypes: float64(5), int64(3), str(8)
memory usage: 865.9 KB


In [5]:
df.isna().sum()

name               0
company            0
model              0
edition            0
year               0
owner              0
fuel               0
seller_type        0
transmission       0
km_driven          0
mileage_mpg      208
engine_cc        208
max_power_bhp    209
torque_nm        209
seats            208
selling_price      0
dtype: int64

In [6]:
df.shape

(6926, 16)

In [7]:
#only  categorical columns
df.select_dtypes(include='O').columns


C:\Users\sudip\AppData\Local\Temp\ipykernel_1124\2836548629.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.select_dtypes(include='O').columns


Index(['name', 'company', 'model', 'edition', 'owner', 'fuel', 'seller_type',
       'transmission'],
      dtype='str')

In [8]:
## here use nunique to find the cardinality of the categorical columns
for col in df.select_dtypes(include='O').columns:
    print(f'column: {col}')
    print(f'cardinality: {df[col].nunique()}')
    print(df[col].unique())
    print(df[col].value_counts(normalize=True))
    print()

column: name
cardinality: 2058
<StringArray>
[                      'Maruti Swift Dzire VDI',
                 'Skoda Rapid 1.5 TDI Ambition',
                     'Honda City 2017-2020 EXi',
                    'Hyundai i20 Sportz Diesel',
                       'Maruti Swift VXI BSIII',
                'Hyundai Xcent 1.2 VTVT E Plus',
                 'Maruti Wagon R LXI DUO BSIII',
                           'Maruti 800 DX BSII',
                             'Toyota Etios VXD',
         'Ford Figo Diesel Celebration Edition',
 ...
                    'Toyota Innova 2.5 E 7 STR',
                            'Honda City ZXi AT',
                 'Ford Fiesta 1.4 Duratorq EXI',
                           'Chevrolet Cruze LT',
                  'Maruti Celerio ZXI AMT BSIV',
                        'Tata Bolt Revotron XM',
           'Tata Manza Aura (ABS) Safire BS IV',
                   'Tata Nexon 1.5 Revotorq XT',
     'Ford Freestyle Titanium Plus Diesel BSIV',
 'Toyota Innova 2.5

C:\Users\sudip\AppData\Local\Temp\ipykernel_1124\960836193.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include='O').columns:


In [9]:
df= df.drop(columns=['name','model','edition'])
df.head()

,company,year,owner,fuel,seller_type,transmission,km_driven,mileage_mpg,engine_cc,max_power_bhp,torque_nm,seats,selling_price
0,Maruti,2014,First,Diesel,Individual,Manual,145500,55.00,1248.0,74.00,190.000000,5.0,450000
1,Skoda,2014,Second,Diesel,Individual,Manual,120000,49.70,1498.0,103.52,250.000000,5.0,370000
2,Honda,2006,Third,Petrol,Individual,Manual,140000,41.60,1497.0,78.00,124.544455,5.0,158000
3,Hyundai,2010,First,Diesel,Individual,Manual,127000,54.06,1396.0,90.00,219.668960,5.0,225000
4,Maruti,2007,First,Petrol,Individual,Manual,120000,37.84,1298.0,88.20,112.776475,5.0,130000


In [10]:
df.duplicated().sum()

np.int64(19)

In [11]:
df=df.drop_duplicates()

In [12]:
df.duplicated().sum()

np.int64(0)

In [17]:
x = df.drop(columns = 'selling_price')
y = df.selling_price.copy()

print(x.shape, y.shape)

(6907, 12) (6907,)


In [20]:
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2, random_state=42)
print(x_train.shape, y_train.shape)
print(x_test.shape, y_test.shape)

(5525, 12) (5525,)
(1382, 12) (1382,)


In [28]:
num_cols=x_train.select_dtypes(include='number').columns.tolist()
cat_cols = [col for col in x_train.columns if col not in num_cols]
print(num_cols)
print(cat_cols)

['year', 'km_driven', 'mileage_mpg', 'engine_cc', 'max_power_bhp', 'torque_nm', 'seats']
['company', 'owner', 'fuel', 'seller_type', 'transmission']


In [ ]:
num_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoder', OneHotEncoder(handle_unknown='ignore',sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])
regressor = RandomForestRegressor(
    n_estimators=10,max_depth=5 ,random_state=42)

re_model = Pipeline(steps=[
    ('pre', preprocessor),
    ('reg', regressor)
])

re_model.fit(x_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('pre', ...), ('reg', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains spar

In [45]:
y_train_pred = re_model.predict(x_train)
train_rmse = root_mean_squared_error(y_train, y_train_pred)
train_mae = mean_absolute_error(y_train, y_train_pred)

y_test_pred = re_model.predict(x_test)
test_rmse = root_mean_squared_error(y_test, y_test_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)

print(f'Train rmse: {train_rmse}')
print()
print(f'Train mae: {train_mae}')
print()
print(f'Test rmse: {test_rmse}')
print()
print(f'Test mae: {test_mae}')

Train rmse: 169947.48964050272

Train mae: 103527.92830970083

Test rmse: 172392.1313605195

Test mae: 105525.95021843615
